In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
import yfinance as yf
from langchain_core.tools import tool
from langchain_tavily import TavilySearch
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

# 1. Financial Statement Tool (NSE & BSE)
@tool
def get_company_financials(symbol: str, exchange: str = "NS") -> str:
    """Fetches key financial statements and available annual profit history.
    Exchange should be 'NS' for NSE or 'BO' for BSE.
    """
    ticker_str = f"{symbol.upper()}.{exchange.upper()}"
    ticker = yf.Ticker(ticker_str)

    income_stmt = ticker.financials
    balance_sheet = ticker.balance_sheet
    cashflow = ticker.cashflow

    if income_stmt.empty:
        return f"No financial data found for {ticker_str}."

    net_income_history = {}
    if "Net Income" in income_stmt.index:
        for period, value in income_stmt.loc["Net Income"].dropna().items():
            net_income_history[str(period.date())] = float(value)

    summary = {
        "ticker": ticker_str,
        "latest_revenue": float(income_stmt.loc["Total Revenue"].iloc[0]) if "Total Revenue" in income_stmt.index else None,
        "latest_net_income": float(income_stmt.loc["Net Income"].iloc[0]) if "Net Income" in income_stmt.index else None,
        "net_income_history": net_income_history,
        "operating_cashflow": float(cashflow.loc["Operating Cash Flow"].iloc[0]) if "Operating Cash Flow" in cashflow.index else None,
        "total_debt": float(balance_sheet.loc["Total Debt"].iloc[0]) if "Total Debt" in balance_sheet.index else None,
    }
    return str(summary)

# 2. Financial News Tool via Tavily
@tool
def get_company_news(company_name: str) -> str:
    """Searches verified financial news sources for recent developments."""
    tavily = TavilySearch(
        max_results=4,
        topic="finance",
        include_domains=["moneycontrol.com", "economictimes.indiatimes.com", "livemint.com"]
    )
    results = tavily.invoke({"query": f"{company_name} quarterly results earnings corporate news"})
    return "\n".join([f"- {r['title']}: {r['content'][:250]}" for r in results.get("results", [])])

api_wrapper_wiki = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=500)

@tool(name_or_callable="wikipedia")
def wikipedia(query: str) -> str:
    """Search Wikipedia for factual background, company founders, executive profiles, and general overviews."""
    return api_wrapper_wiki.run(query)

In [ ]:
### Combine all the tools in the list

tools = [get_company_financials, get_company_news, wikipedia]

In [ ]:
# Test the tools directly before testing the LangGraph
import os

print("Tavily API key loaded:", bool(os.getenv("TAVILY_API_KEY")))

print("--- Infosys financials ---")
try:
    financials_result = get_company_financials.invoke({
        "symbol": "INFY",
        "exchange": "NS",
    })
    print(financials_result)
except Exception as error:
    print(f"Financials tool failed: {type(error).__name__}: {error}")

print("\n--- Infosys news ---")
try:
    news_result = get_company_news.invoke({
        "company_name": "Infosys",
    })
    print(news_result or "No news results returned.")
except Exception as error:
    print(f"News tool failed: {type(error).__name__}: {error}")

In [ ]:
## Initialize my LLM model
import os
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    groq_api_key=os.getenv("GROQ_API_KEY"),
    max_tokens=1000,
    temperature=0
)

# Ask for at most one tool call in the tool-selection step.
llm_with_tools = llm.bind_tools(tools, parallel_tool_calls=False)

In [ ]:
from pprint import pprint
from langchain_core.messages import AIMessage, HumanMessage


answer = llm_with_tools.invoke([HumanMessage(content=f"What is the recent news about infosys")])

In [ ]:
answer

In [ ]:
answer.pretty_print()

In [ ]:
answer.tool_calls

In [ ]:
from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage
from typing import Annotated, List, NotRequired
from langgraph.graph.message import add_messages

class FinanceState(TypedDict):
    messages: Annotated[List[AnyMessage], add_messages]
    summary: NotRequired[str]

In [ ]:
from langchain_core.messages import trim_messages

def tool_calling_llm_trim(state:FinanceState):
    trimmed = trim_messages(
        state["messages"],
        max_tokens=700,
        strategy="last",
        token_counter="approximate",
        start_on="human",          # LLMs usually require history to start with a user message
        include_system=True        # Preserves any top-level instructions
    )
    return {"messages": [llm_with_tools.invoke(trimmed)]}

In [ ]:
def tool_calling_llm_last_5(state: FinanceState):
    # Slice the last 4 messages, or the whole list if shorter
    recent_messages = state["messages"][-4:]
    return {"messages": [llm_with_tools.invoke(recent_messages)]}

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, RemoveMessage

def summarize_conversation(state: FinanceState):
    summary = state.get("summary", "")
    messages = state["messages"]
    
    if len(messages) > 2:
        cutoff = len(messages) - 2
        older_messages = messages[:cutoff]

        system_instruction = SystemMessage(
            content="You are a conversation summarizer. Summarize prior discussions in plain text only. "
                    "Do NOT attempt to use tools, call functions, or output JSON."
        )

        prompt = (
            f"Existing summary:\n{summary}\n\n" if summary else ""
        ) + "Summarize the key facts, user preferences, and companies mentioned in these messages:"

        new_summary = llm.invoke([system_instruction, HumanMessage(content=prompt)] + older_messages)
        delete_messages = [RemoveMessage(id=m.id) for m in older_messages]
        
        return {
            "summary": new_summary.content,
            "messages": delete_messages
        }
    return {}

In [ ]:
from langchain_core.messages import SystemMessage

def tool_calling_llm_summarize(state: FinanceState):
    summary = state.get("summary", "")
    messages = state["messages"]
    
    if summary:
        system_msg = SystemMessage(content=f"Summary of earlier conversation:\n{summary}")
        messages = [system_msg] + messages
        
    return {"messages": [llm_with_tools.invoke(messages)]}

In [ ]:
tool_calling_llm = tool_calling_llm_summarize

In [ ]:
from langchain_core.messages import SystemMessage


def summarize_tool_result(state: FinanceState):
    system_message = SystemMessage(content=(
        "Answer only from the retrieved tool result. Do not invent values, dates, "
        "growth rates, or news. If the requested history is not present, say which "
        "periods are unavailable. The tool result is informational, not investment advice."
    ))
    return {"messages": [llm.invoke([system_message] + state["messages"])]}

In [ ]:
# Build graph with Checkpointer and Summarization routing
from typing import Literal
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import InMemorySaver
from IPython.display import Image, display

# 1. Routing function from tool_calling_llm
def route_from_llm(state: FinanceState) -> Literal["tools", "summarize_conversation", "__end__"]:
    last_message = state["messages"][-1]
    
    # If the model requested tool calls, route directly to tools node
    if getattr(last_message, "tool_calls", None):
        return "tools"
    
    # If it was a direct text reply, check if conversation backlog > 2
    if len(state.get("messages", [])) > 2:
        return "summarize_conversation"
    
    return END

# 2. Routing function after tool summary is completed
def should_summarize(state: FinanceState) -> Literal["summarize_conversation", "__end__"]:
    if len(state.get("messages", [])) > 2:
        return "summarize_conversation"
    return END

builder = StateGraph(FinanceState)

# 3. Register all nodes
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools", ToolNode(tools))
builder.add_node("summarize_tool_result", summarize_tool_result)
builder.add_node("summarize_conversation", summarize_conversation)

# 4. Define control flow edges
builder.add_edge(START, "tool_calling_llm")

# Conditional edge from LLM node (handles both tools and direct conversation)
builder.add_conditional_edges(
    "tool_calling_llm",
    route_from_llm,
    {
        "tools": "tools",
        "summarize_conversation": "summarize_conversation",
        END: END
    }
)

# After running tools, synthesize results
builder.add_edge("tools", "summarize_tool_result")

# After synthesizing results, check if summarization is needed
builder.add_conditional_edges(
    "summarize_tool_result",
    should_summarize,
    {
        "summarize_conversation": "summarize_conversation",
        END: END
    }
)

# Conclude after summarizing
builder.add_edge("summarize_conversation", END)

# 5. Compile with in-memory checkpointer
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
# Turn 1: Starts thread and fetches news
config = {"configurable": {"thread_id": "finance_thread_01"}}

result1 = graph.invoke(
    {"messages": [HumanMessage(content="Infosys is my favourite company to track financial information but i dont know what it does under 100 words? ")]},
    config=config
)

for message in result1.get("messages", []):
    message.pretty_print()

state1 = graph.get_state(config)
print("\n--- STATE STATUS (TURN 1) ---")
print(f"Messages in memory: {len(state1.values['messages'])}")
print(f"Summary exists: {bool(state1.values.get('summary'))}")

In [ ]:
# Turn 2: Follow-up question under the SAME thread. 
# Total messages exceed cutoff of 2 -> triggers summarize_conversation.
result2 = graph.invoke(
    {"messages": [HumanMessage(content="What is the profit growth of infosys of last 3 years?")]},
    config=config
)

for message in result2.get("messages", []):
    message.pretty_print()

state2 = graph.get_state(config)
print("\n--- STATE STATUS (TURN 2) ---")
print(f"Messages remaining in memory: {len(state2.values['messages'])}")
print("Active Summary stored in State:")
print(state2.values.get("summary"))

In [ ]:
# Turn 2: Follow-up question under the SAME thread. 
# Total messages exceed cutoff of 2 -> triggers summarize_conversation.
result3 = graph.invoke(
    {"messages": [HumanMessage(content="Which is my favourite company to track?")]},
    config=config
)

for message in result3.get("messages", []):
    message.pretty_print()

state3 = graph.get_state(config)
print("\n--- STATE STATUS (TURN 3) ---")
print(f"Messages remaining in memory: {len(state3.values['messages'])}")
print("Active Summary stored in State:")
print(state3.values.get("summary"))